In [ ]:
import pandas as pd
import geopandas as gpd
from datetime import datetime

import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

In [ ]:
# 1. Calculate HBE Rate per segment
# The HBE rate is measured as HBEs per million vehicle miles traveled:
#   hbe_count / (AADT * length_mi * days_covered) * 1,000,000
# Counts come from hbe_counts.py (full export, braking events only, 75 ft flat-cap buffers,
# each event assigned to the one nearest segment whose direction matches its heading).
HBE_COUNTS_CSV = "hbe_work_heading/segment_hbe_counts.csv"  # heading-matched run; hbe_work/ = every-buffer run
hbe_counts = pd.read_csv(
    HBE_COUNTS_CSV,
    usecols=["CTN_SEGMENT_ID", "hbe_count", "days_covered"],
)
days_covered = int(hbe_counts["days_covered"].iloc[0])
print(f"{HBE_COUNTS_CSV}: {len(hbe_counts):,} segments, {hbe_counts['hbe_count'].sum():,} event-segment matches, {days_covered} days")

In [ ]:
ctn = gpd.read_file("data/ctn_within_boundary.geojson")

# Transforming to projected coordinates
# NAD83 / Texas Central (ftUS)
ctn = ctn.to_crs(epsg=2277)

# Buffer the lines by 75 feet with FLAT caps (no rounded/square ends)
# Still needed below for the crash spatial join
line_buffers = ctn.copy()
line_buffers["geometry"] = ctn.buffer(75, cap_style="flat")
line_geoms = ctn.geometry.copy()  # centerlines, same index as line_buffers

# Attach the precomputed HBE counts to the lines layer
hbe_counts["CTN_SEGMENT_ID"] = hbe_counts["CTN_SEGMENT_ID"].astype(ctn["CTN_SEGMENT_ID"].dtype)
ctn = ctn.merge(hbe_counts[["CTN_SEGMENT_ID", "hbe_count"]], on="CTN_SEGMENT_ID", how="left")
ctn["hbe_count"] = ctn["hbe_count"].fillna(0).astype(int)
assert len(ctn) == len(line_buffers), "CTN_SEGMENT_ID is not unique in the CTN layer"

In [ ]:
# Getting AADT
aadts = pd.read_csv("data/24_aadt_manual.csv")

In [ ]:
ctn = ctn.merge(aadts, left_on="CTN_SEGMENT_ID", right_on="Zone Name", how="left")

In [ ]:
ctn["length_ft"] = ctn.length
ctn["length_mi"] = ctn["length_ft"] / 5280

In [ ]:
ctn["vmt"] = ctn["Estimated 2024 AADT (initial release)"] * ctn["length_mi"]

In [ ]:
ctn["hbe_rate"] = (ctn["hbe_count"] / (ctn["vmt"] * days_covered) * 1_000_000).replace([np.inf, -np.inf], np.nan)

In [ ]:
# 2. Calculate Crash Rate per segment
# "Crash rate is measured as crashes per million vehicle miles traveled (i.e., number of
# crashes/(AADT*365* years*length of the segment)/1,000,000)."

In [ ]:
data_url = "https://data.austintexas.gov/api/v3/views/y2wy-tgr5/query.geojson?$limit=9999999"

In [ ]:
crashes = gpd.read_file(data_url)

In [ ]:
import shapely

# Transforming to projected coordinates
# NAD83 / Texas Central (ftUS)
crashes = crashes.to_crs(epsg=2277)
assert crashes.crs == ctn.crs
assert len(ctn) == len(line_buffers), "ctn rows no longer line up with line_buffers (duplicate join keys?)"

# Assign each crash to ONE segment, consistent with the HBE matching:
# candidates = buffers the crash falls within; winner = nearest centerline.
# Crash records have no heading, so distance alone decides.
cand = gpd.sjoin(crashes[["geometry"]], line_buffers[["geometry"]], predicate="within", how="inner")
cand["dist_ft"] = shapely.distance(
    np.asarray(cand.geometry.values),
    np.asarray(line_geoms.loc[cand["index_right"]].values),
)
nearest = cand.sort_values("dist_ft")
nearest = nearest[~nearest.index.duplicated(keep="first")]
print(f"{len(crashes):,} crashes, {len(nearest):,} inside a buffer, "
      f"{cand.index.duplicated().sum():,} extra matches removed (were double counted before)")

counts = nearest.groupby("index_right").size().rename("crash_count")

# Attach the counts back to the lines layer
ctn = ctn.join(counts)
ctn["crash_count"] = ctn["crash_count"].fillna(0).astype(int)

In [ ]:
start = datetime(2010, 1, 1)
now = datetime.now()

years_passed = (now - start).days / 365.25

In [ ]:
ctn["crash_rate"] = (ctn["crash_count"] / ((365 * years_passed * ctn["vmt"])/1000000)).replace([np.inf, -np.inf], np.nan)

In [ ]:
ctn_meta = pd.read_csv("data/ctn_metadata.csv")
ctn = ctn.merge(ctn_meta, left_on="CTN_SEGMENT_ID", right_on="CTN_SEGMENT_ID", how="left")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Binned crash rate vs HBE rate, following Li et al. (arXiv 2601.06327) Fig. 4:
#   - segments ranked by HBE rate within each street level, 10 equal-count bins
#   - zero-HBE segments kept, as their own bin, drawn at a fixed spot left of the axis
#   - zero-crash segments kept (they're real information, not missing data)
# Same segment filters as the NB model cell: roadway, > 300 ft, AADT > 500.
# Left panel: mean of per-segment rates (the paper's method).
# Right panel: pooled rate per bin (total events / total VMT), which short,
#              low-volume segments can't dominate.

AADT = "Estimated 2024 AADT (initial release)"
level_names = {
    1: "Level 1 – Local",
    2: "Level 2 – Collector",
    3: "Level 3 – Minor arterial",
    4: "Level 4 – Major arterial / frontage",
    5: "Level 5 – Freeway",
}
# Validated categorical palette for a dark surface (all pairs pass CVD + 3:1 contrast)
colors = dict(zip(level_names, ["#338ced", "#e76624", "#00a888", "#d28e05", "#e24d8b"]))
markers = dict(zip(level_names, ["d", "^", "x", "s", "o"]))

df = ctn[
    (ctn["FACILITY_TYPE"] == "Roadway")
    & (ctn["length_ft"] > 300)
    & (ctn[AADT] > 500)
    & ctn["STREET_LEVEL"].isin(level_names.keys())
].dropna(subset=["hbe_rate", "crash_rate"]).copy()

df["crash_exposure_mvmt"] = 365 * years_passed * df["vmt"] / 1e6  # crash window
df["hbe_exposure_mvmt"] = days_covered * df["vmt"] / 1e6           # HBE window

# Bin -1 = zero-HBE segments; 0..9 = deciles of the positive-HBE segments, per level
df["hbe_bin"] = -1
pos = df["hbe_rate"] > 0
df.loc[pos, "hbe_bin"] = (
    df[pos].groupby("STREET_LEVEL")["hbe_rate"]
    .transform(lambda x: pd.qcut(x.rank(method="first"), q=10, labels=False))
    .astype(int)
)

binned = df.groupby(["STREET_LEVEL", "hbe_bin"]).agg(
    n_segments=("hbe_rate", "size"),
    pct_zero_crash=("crash_count", lambda s: round(100 * (s == 0).mean(), 1)),
    mean_hbe_rate=("hbe_rate", "mean"),
    mean_crash_rate=("crash_rate", "mean"),
    hbe_sum=("hbe_count", "sum"),
    hbe_exp=("hbe_exposure_mvmt", "sum"),
    crash_sum=("crash_count", "sum"),
    crash_exp=("crash_exposure_mvmt", "sum"),
).reset_index()
binned["pooled_hbe_rate"] = binned["hbe_sum"] / binned["hbe_exp"]
binned["pooled_crash_rate"] = binned["crash_sum"] / binned["crash_exp"]


def safe_log(s):
    return np.log(s.where(s > 0))  # a bin with zero crashes has no log; leave it out


# x position for the zero-HBE bin: a bit left of the smallest positive bin
positive = binned[binned["hbe_bin"] >= 0]
ZERO_X = np.floor(np.log(positive[["mean_hbe_rate", "pooled_hbe_rate"]].min().min())) - 1.5

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5), sharex=True, sharey=True)
panels = [
    ("mean_hbe_rate", "mean_crash_rate", "Mean of segment rates (paper's method)"),
    ("pooled_hbe_rate", "pooled_crash_rate", "Pooled rate per bin (total ÷ VMT)"),
]
for ax, (xcol, ycol, title) in zip(axes, panels):
    for level, name in level_names.items():
        g = binned[binned["STREET_LEVEL"] == level].sort_values("hbe_bin")
        if g.empty:
            continue
        zero, deciles = g[g["hbe_bin"] == -1], g[g["hbe_bin"] >= 0]
        x, y = safe_log(deciles[xcol]), safe_log(deciles[ycol])
        ax.plot(x, y, marker=markers[level], color=colors[level],
                linewidth=2, markersize=8, label=name)
        if not zero.empty and not deciles.empty:
            zy = safe_log(zero[ycol]).iloc[0]
            # hollow marker + dashed link = the zero-HBE bin
            ax.plot([ZERO_X, x.iloc[0]], [zy, y.iloc[0]], linestyle=":",
                    color=colors[level], linewidth=1.5)
            ax.plot(ZERO_X, zy, marker=markers[level], markersize=9,
                    markerfacecolor="none", markeredgecolor=colors[level], markeredgewidth=1.5)
    ax.axvline(ZERO_X + 0.75, color="gray", linewidth=0.8, alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel("log(HBEs per million VMT)")
    ax.grid(alpha=0.15)

ticks = [t for t in axes[0].get_xticks() if t > ZERO_X + 0.75]
axes[0].set_xticks([ZERO_X] + ticks)
axes[0].set_xticklabels(["0 HBE"] + [f"{t:g}" for t in ticks])
axes[0].set_ylabel("log(crashes per million VMT)")
axes[0].legend(loc="upper left")
fig.suptitle(f"Austin TX — {days_covered} days of HBE data, {len(df):,} segments "
             f"(roadway, > 300 ft, AADT > 500)")
plt.tight_layout()
plt.show()

summary = df.groupby("STREET_LEVEL").agg(
    segments=("hbe_rate", "size"),
    pct_zero_hbe=("hbe_rate", lambda s: round(100 * (s == 0).mean(), 1)),
    pct_zero_crash=("crash_count", lambda s: round(100 * (s == 0).mean(), 1)),
)
summary.index = summary.index.map(level_names)
print(summary)
print(binned[["STREET_LEVEL", "hbe_bin", "n_segments", "pct_zero_crash",
              "mean_hbe_rate", "mean_crash_rate", "pooled_hbe_rate", "pooled_crash_rate"]]
      .round(3).to_string(index=False))

In [ ]:
# Export the highest-HBE segments for mapping, next to the deciles just below them,
# with the attributes that might explain why the top decile has fewer crashes per VMT.
# Uses `df` and its per-level `hbe_bin` from the plot cell above (0-9 = deciles 1-10).

signals = gpd.read_file(
    "https://data.austintexas.gov/api/v3/views/p53x-x73x/query.geojson?$limit=9999999"
).to_crs(epsg=2277)
for col in ("signal_type", "signal_status"):
    if col in signals.columns:
        print(signals[col].value_counts(dropna=False), "\n")
# The dataset includes pedestrian hybrid beacons; to keep only full traffic signals,
# filter on the type column printed above, e.g.:
# signals = signals[signals["signal_type"] == "TRAFFIC"]

top = df[df["hbe_bin"] >= 6].copy()  # deciles 7-10 within each street level
top["decile"] = top["hbe_bin"] + 1
top["group"] = np.where(top["decile"] == 10, "top decile", "deciles 7-9")

# Distance from each segment (anywhere along it) to the nearest signal
top = gpd.sjoin_nearest(top, signals[["geometry"]], how="left", distance_col="dist_to_signal_ft")
top = top[~top.index.duplicated(keep="first")].drop(columns="index_right")
top["near_signal"] = top["dist_to_signal_ft"] <= 150
top = top.rename(columns={"Estimated 2024 AADT (initial release)": "aadt_2024"})

attrs = [c for c in ["length_ft", "aadt_2024", "SPEED_LIMIT", "TOTAL_LANE_COUNT",
                     "dist_to_signal_ft", "hbe_rate", "crash_rate"] if c in top.columns]
summary = top.groupby(["STREET_LEVEL", "group"])[attrs].median()
summary["pct_near_signal"] = top.groupby(["STREET_LEVEL", "group"])["near_signal"].mean() * 100
summary["segments"] = top.groupby(["STREET_LEVEL", "group"]).size()
print("Medians (pct_near_signal = share within 150 ft of a signal):")
print(summary.round(1).to_string())

keep = ["CTN_SEGMENT_ID", "FULL_STREET_NAME", "STREET_LEVEL", "decile", "group",
        *attrs, "near_signal", "hbe_count", "crash_count", "geometry"]
out = top[[c for c in keep if c in top.columns]].to_crs(epsg=4326)
out.to_file("top_hbe_segments.geojson", driver="GeoJSON")
print(f"\nWrote top_hbe_segments.geojson ({len(out):,} segments)")

In [ ]:
top5 = ctn[ctn["length_ft"] > 300]
top5 = (
    top5.groupby("STREET_LEVEL", group_keys=True)
    .apply(lambda x: x.nlargest(5, "hbe_rate"))
)

top5 = top5.reset_index(level=0)  # brings STREET_LEVEL back as a column, drops the leftover row-index level
top5 = top5.reset_index(drop=True)  # clean up the remaining index

print(top5[["STREET_LEVEL", "CTN_SEGMENT_ID", "hbe_rate"]])

In [ ]:
out = ctn.to_crs(epsg=4326)
out["log_hbe_rate"] = np.log(out["hbe_rate"])
out["log_crash_rate"] = np.log(out["crash_rate"])
out.to_file("hard_brake_crash_rates.geojson", driver="GeoJSON", index=False)

In [ ]:
AADT = "Estimated 2024 AADT (initial release)"

ctn = ctn[ctn["length_ft"] > 300]
ctn = ctn[ctn[AADT] > 500]
ctn["exposure"] = ctn[AADT] * ctn["length_mi"] * years_passed

# Drop segments missing critical info. TOTAL_LANE_COUNT is left out of the model because
# it's sparsely populated (requiring it dropped most freeway and many arterial segments).
ctn = ctn.dropna(subset=[AADT, "length_mi", "STREET_LEVEL", "SPEED_LIMIT", "SLOPE"])
ctn = ctn[ctn["exposure"] > 0]
ctn = ctn.dropna(subset=["hbe_rate"])  # only drop true missingness, not zeros

# log HBE rate (elasticity); zero-HBE segments get 0 plus their own indicator
ctn["zero_hbe"] = (ctn["hbe_rate"] == 0).astype(int)
ctn["log_hbe_rate"] = np.log(ctn["hbe_rate"].where(ctn["hbe_rate"] > 0, 1.0))

# AADT as its own term. With exposure (which includes AADT) as the offset, crashes scale
# with AADT^(1 + coef): coef = 0 means 1:1 with volume, negative means less than 1:1.
# It also tests whether the HBE effect survives once AADT, which hbe_rate is divided by,
# is in the model separately.
ctn["log_aadt"] = np.log(ctn[AADT])

X = pd.get_dummies(ctn[["STREET_LEVEL", "SPEED_LIMIT", "SLOPE", "log_aadt",
                        "log_hbe_rate", "zero_hbe"]],
                   columns=["STREET_LEVEL"], drop_first=True)
X = sm.add_constant(X).astype(float)

y = ctn["crash_count"].astype(float)
log_exposure = np.log(ctn["exposure"])


def fit_nb(X):
    """Poisson start, then negative binomial with Newton's method."""
    p = sm.GLM(y, X, family=sm.families.Poisson(), offset=log_exposure).fit()
    return sm.NegativeBinomial(y, X, offset=log_exposure).fit(
        start_params=np.append(p.params.values, 0.1), method="newton", maxiter=200, disp=False)


nb_model_hbe = fit_nb(X)
print("converged:", nb_model_hbe.mle_retvals["converged"])
print(f"{len(ctn):,} segments in the model")
print(nb_model_hbe.summary())

b = nb_model_hbe.params["log_hbe_rate"]
lo, hi = nb_model_hbe.conf_int().loc["log_hbe_rate"]
print(f"\nHBE elasticity {b:.3f} (95% CI {lo:.3f}–{hi:.3f}): doubling a segment's HBE rate "
      f"is associated with {100 * (2**b - 1):.0f}% more crashes, holding the other terms fixed.")
print(f"Implied AADT exponent: {1 + nb_model_hbe.params['log_aadt']:.3f} "
      f"(crash count scales with AADT to this power)")

# Same model without the HBE terms, to measure what HBEs add
nb0 = fit_nb(X.drop(columns=["log_hbe_rate", "zero_hbe"]))
print(f"\nLR statistic: {2 * (nb_model_hbe.llf - nb0.llf):,.0f} on 2 df")
print(f"AIC without HBE: {nb0.aic:,.0f}   with HBE: {nb_model_hbe.aic:,.0f}")

In [ ]:
ctn["predicted_log_rate"] = nb_model_hbe.predict(X, offset=np.zeros(len(X)))
ctn_ranked = ctn.sort_values("predicted_log_rate", ascending=False)
ctn_ranked[["FULL_STREET_NAME", "STREET_LEVEL", "hbe_rate", "predicted_log_rate", "crash_count"]].head(20)

In [ ]:
print(ctn.groupby("HIN")["predicted_log_rate"].mean())

In [ ]:
out = ctn.to_crs(epsg=4326)
out.to_file("predicted_hbe_rates.geojson", driver="GeoJSON", index=False)

In [ ]:
# Network screening: readable rates, Empirical Bayes (EB) expected crashes, excess crashes,
# a check against the High Injury Network, an interactive map, and geojson.io exports.
# Run after the model cell (uses nb_model_hbe, X, ctn, log_exposure, years_passed).
# The map needs folium and mapclassify:  pip install folium mapclassify
import folium
import shapely

AADT = "Estimated 2024 AADT (initial release)"
alpha = nb_model_hbe.params["alpha"]
s = ctn.copy()

# --- 1. Model predictions in readable units ---------------------------------------------
# predict() with a zero offset gives crashes per unit of exposure (AADT x miles x years);
# x 1e6 / 365 converts that to crashes per million vehicle miles.
s["pred_rate_mvmt"] = nb_model_hbe.predict(X, offset=np.zeros(len(X))) * 1e6 / 365
s["pred_crashes"] = nb_model_hbe.predict(X, offset=log_exposure)  # over the crash window

# --- 2. Empirical Bayes (Highway Safety Manual) -----------------------------------------
# Blend each segment's prediction with its own crash history. The weight on the prediction
# shrinks as expected crashes grow, so busy segments lean on their history and short or
# quiet ones lean on the model.
w = 1 / (1 + alpha * s["pred_crashes"])
s["eb_weight"] = w
s["eb_crashes"] = w * s["pred_crashes"] + (1 - w) * s["crash_count"]
s["excess_crashes"] = s["eb_crashes"] - s["pred_crashes"]  # > 0: worse than similar roads

mi_yr = s["length_mi"] * years_passed
s["obs_per_mi_yr"] = s["crash_count"] / mi_yr
s["pred_per_mi_yr"] = s["pred_crashes"] / mi_yr
s["eb_per_mi_yr"] = s["eb_crashes"] / mi_yr
s["excess_per_mi_yr"] = s["excess_crashes"] / mi_yr

print(f"alpha = {alpha:.3f}; median EB weight on the model = {w.median():.2f}")
print(f"Observed crashes {s['crash_count'].sum():,.0f}, predicted {s['pred_crashes'].sum():,.0f}, "
      f"EB {s['eb_crashes'].sum():,.0f} over {years_passed:.1f} years")

# --- 3. Top segments and the High Injury Network check ------------------------------------
cols = ["FULL_STREET_NAME", "STREET_LEVEL", AADT, "length_ft", "crash_count",
        "pred_rate_mvmt", "eb_per_mi_yr", "excess_per_mi_yr", "HIN"]
top20 = s.sort_values("excess_per_mi_yr", ascending=False)[[c for c in cols if c in s.columns]].head(20)
print("\nTop 20 segments by excess expected crashes per mile per year:")
print(top20.round(2).to_string())

if "HIN" in s.columns:
    on_hin = s["HIN"].eq("Y")
    def hin_share(mask):
        return 100 * s.loc[mask & on_hin, "length_mi"].sum() / s.loc[mask, "length_mi"].sum()
    everything = pd.Series(True, index=s.index)
    print("\nShare of segment miles on the High Injury Network:")
    print(f"  all modeled segments:                {hin_share(everything):5.1f}%")
    for col, label in [("excess_per_mi_yr", "excess crashes / mi / yr"),
                       ("eb_per_mi_yr", "EB crashes / mi / yr"),
                       ("pred_rate_mvmt", "predicted rate / MVMT")]:
        top5 = s[col] >= s[col].quantile(0.95)
        print(f"  top 5% by {label:<25} {hin_share(top5):5.1f}%")

# --- 4. Color classes -------------------------------------------------------------------
SEQ = ["#b6d5f8", "#6aabf1", "#338ced", "#1b64be", "#0e3d7e"]   # light -> dark blue
DIV = ["#1b64be", "#84b9f3", "#c8c8c8", "#f08a8b", "#c42b2c"]   # blue <- gray -> red


def quantile_classes(x, colors):
    """Five equal-count classes, labelled with their value ranges."""
    cats, edges = pd.qcut(x, len(colors), retbins=True, duplicates="drop")
    labels = [f"{edges[i]:,.2f} – {edges[i + 1]:,.2f}" for i in range(len(edges) - 1)]
    return pd.Series(np.array(labels)[cats.cat.codes], index=x.index), labels, colors[:len(labels)]


s["rate_class"], rate_labels, rate_colors = quantile_classes(s["pred_rate_mvmt"], SEQ)
s["eb_class"], eb_labels, eb_colors = quantile_classes(s["eb_per_mi_yr"], SEQ)

# Excess is signed, so it gets symmetric diverging classes around zero
t1, t2 = s["excess_per_mi_yr"].abs().quantile([0.5, 0.9])
exc_labels = [f"much better than predicted (< −{t2:.2f})", f"better (−{t2:.2f} to −{t1:.2f})",
              f"about as predicted (±{t1:.2f})", f"worse (+{t1:.2f} to +{t2:.2f})",
              f"much worse than predicted (> +{t2:.2f})"]
s["excess_class"] = pd.cut(s["excess_per_mi_yr"], [-np.inf, -t2, -t1, t1, t2, np.inf],
                           labels=exc_labels).astype(str)

layers = [  # (layer name, class column, labels, colors, file name)
    ("Excess expected crashes / mi / yr (EB − predicted)", "excess_class", exc_labels, DIV,
     "screening_excess.geojson"),
    ("EB expected crashes / mi / yr", "eb_class", eb_labels, eb_colors, "screening_eb.geojson"),
    ("Predicted crash rate / million VMT", "rate_class", rate_labels, rate_colors,
     "screening_rate.geojson"),
]

# --- 5. Interactive map (opens in the notebook; also saved as HTML) -------------------------
web = s.assign(geometry=s.geometry.simplify(5)).to_crs(epsg=4326)  # 5 ft simplify keeps the file light
web = web.rename(columns={AADT: "aadt_2024"})
tip = [c for c in ["FULL_STREET_NAME", "STREET_LEVEL", "aadt_2024", "crash_count", "pred_rate_mvmt",
                   "eb_per_mi_yr", "excess_per_mi_yr", "HIN"] if c in web.columns]
for c in ["pred_rate_mvmt", "eb_per_mi_yr", "excess_per_mi_yr", "eb_weight"]:
    web[c] = web[c].round(2)
# Keep only what the map and exports use (every column is embedded in each map layer)
web = web[["CTN_SEGMENT_ID", *tip, "eb_weight", *[l[1] for l in layers], "geometry"]]
web["geometry"] = shapely.set_precision(web.geometry.values, 1e-6)  # ~4 inches

m = None
for i, (name, col, labels, colors, _) in enumerate(layers):
    m = web.explore(
        m=m, column=col, categorical=True, categories=labels, cmap=colors,
        name=name, show=(i == 0), legend=(i == 0),  # one legend; others listed in the layer control
        tooltip=tip, style_kwds={"weight": 3}, tiles=None,
        legend_kwds={"caption": name},
    )
folium.TileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, © OpenStreetMap contributors", name="Esri Light Gray",
).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save("safety_screening_map.html")

# --- 6. geojson.io exports: one file per measure, colored with the same classes -------------
for name, col, labels, colors, fname in layers:
    g = web[["CTN_SEGMENT_ID", *tip, "eb_weight", col, "geometry"]].copy()
    g["stroke"] = g[col].map(dict(zip(labels, colors)))
    g["stroke-width"] = 3
    g.to_file(fname, driver="GeoJSON", COORDINATE_PRECISION=6)
print("\nWrote safety_screening_map.html and", ", ".join(l[4] for l in layers))


s["excess_per_yr"] = s["excess_crashes"] / years_passed
s["hbe_per_mi"] = s["hbe_count"] / s["length_mi"] if "hbe_count" in s.columns else np.nan
print("\nShare of segment miles on the HIN, top 5% by:")
for col, label in [("pred_per_mi_yr", "predicted crashes / mi / yr (model only)"),
                   ("hbe_rate", "HBE rate alone"),
                   ("hbe_per_mi", "HBEs per mile alone"),
                   ("excess_per_yr", "excess crashes / yr (no per-mile)")]:
    if s[col].notna().any():
        top5 = s[col] >= s[col].quantile(0.95)
        print(f"  {label:<42} {hin_share(top5):5.1f}%")
print(f"\nTop-5% excess (per mile) segments under 500 ft: "
      f"{100 * (s.loc[s['excess_per_mi_yr'] >= s['excess_per_mi_yr'].quantile(0.95), 'length_ft'] < 500).mean():.0f}%"
      f"  vs all segments: {100 * (s['length_ft'] < 500).mean():.0f}%")

m